# CBRAIN OASIS Cohort Construction (Step 2A)

This notebook builds a deterministic baseline cohort and freezes leakage-safe datasets for modeling.

In [ ]:
# 1) Load raw CSV from data/oasis_dementia_mri_df.csv
from __future__ import annotations

import csv
import hashlib
import json
from datetime import date
from pathlib import Path

cwd = Path.cwd().resolve()
candidate_roots = [cwd]
if cwd.name == 'notebooks':
    candidate_roots.append(cwd.parent)
else:
    candidate_roots.append(cwd / 'projects' / 'cbrain_oasis_cognition')
    candidate_roots.append(cwd.parent)

repo_root = None
for candidate in candidate_roots:
    csv_path_candidate = candidate / 'data' / 'oasis_dementia_mri_df.csv'
    if csv_path_candidate.exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError('Could not locate data/oasis_dementia_mri_df.csv from current working directory.')

data_path = (repo_root / 'data' / 'oasis_dementia_mri_df.csv').resolve()
results_dir = (repo_root / 'results' / 'cohort').resolve()
docs_path = (repo_root / 'docs' / 'cohort_construction.md').resolve()

with data_path.open('r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    raw_columns = list(reader.fieldnames or [])
    rows = [dict(row) for row in reader]

if not rows:
    raise ValueError('Input CSV has no data rows.')

print(f'Repo root: {repo_root}')
print(f'Loaded {len(rows)} rows and {len(raw_columns)} columns from {data_path.name}')

In [ ]:
# 2) Build binary label cognitive_impairment = int(CDR > 0)
for row in rows:
    cdr_value = float(row['CDR'])
    row['cognitive_impairment'] = int(cdr_value > 0.0)

label_values = {row['cognitive_impairment'] for row in rows}
if not label_values.issubset({0, 1}):
    raise AssertionError(f'Label domain invalid: {label_values}')

if any(row['cognitive_impairment'] is None for row in rows):
    raise AssertionError('Label contains missing values.')

print('Label domain:', sorted(label_values))

In [ ]:
# 3) Construct baseline cohort (one row per Subject.ID)
def _to_int(value: str) -> int:
    return int(value)

rows_sorted_for_baseline = sorted(
    rows,
    key=lambda r: (
        r['Subject.ID'],
        _to_int(r['Visit']),
        _to_int(r['MR.Delay']),
        r['MRI.ID'],
    ),
)

baseline_rows = []
seen_subject_ids = set()
for row in rows_sorted_for_baseline:
    subject_id = row['Subject.ID']
    if subject_id in seen_subject_ids:
        continue
    seen_subject_ids.add(subject_id)
    baseline_rows.append(dict(row))

baseline_rows = sorted(baseline_rows, key=lambda r: r['Subject.ID'])
raw_subject_ids = sorted({row['Subject.ID'] for row in rows})

if len(baseline_rows) != len(raw_subject_ids):
    raise AssertionError('Baseline cohort row count does not match unique subject count.')

if len({row['Subject.ID'] for row in baseline_rows}) != len(baseline_rows):
    raise AssertionError('Duplicate Subject.ID found in baseline cohort.')

print(f'Baseline rows: {len(baseline_rows)} (unique subjects: {len(raw_subject_ids)})')

In [ ]:
# 4) Define locked exclusions for main model
locked_exclusions = [
    'CDR',
    'Group',
    'MMSE',
    'Subject.ID',
    'MRI.ID',
    'rownames',
    'Hand',
    'Visit',
    'MR.Delay',
]

print('Locked exclusions:', locked_exclusions)

In [ ]:
# 5) Define baseline v1 candidate predictors
candidate_predictors = ['Age', 'Gender', 'EDUC', 'SES', 'eTIV', 'nWBV', 'ASF']

missing_predictors = [col for col in candidate_predictors if col not in raw_columns]
if missing_predictors:
    raise AssertionError(f'Missing required predictor columns: {missing_predictors}')

excluded_predictors = [col for col in candidate_predictors if col in set(locked_exclusions)]
if excluded_predictors:
    raise AssertionError(f'Predictors overlap with exclusions: {excluded_predictors}')

baseline_model_rows = []
for row in baseline_rows:
    model_row = {col: row[col] for col in candidate_predictors}
    model_row['cognitive_impairment'] = row['cognitive_impairment']
    baseline_model_rows.append(model_row)

model_columns = list(baseline_model_rows[0].keys())
for leakage_col in locked_exclusions:
    if leakage_col in model_columns:
        raise AssertionError(f'Leakage column present in model table: {leakage_col}')

print('Model columns:', model_columns)

In [ ]:
# 6) Create deterministic subject split using sha256(subject_id) % 10
def subject_bucket(subject_id: str) -> int:
    digest = hashlib.sha256(subject_id.encode('utf-8')).hexdigest()
    return int(digest, 16) % 10

train_subject_ids = sorted([sid for sid in raw_subject_ids if subject_bucket(sid) < 8])
validation_subject_ids = sorted([sid for sid in raw_subject_ids if subject_bucket(sid) >= 8])

subject_overlap = sorted(set(train_subject_ids).intersection(validation_subject_ids))
overlap_count = len(subject_overlap)
if overlap_count != 0:
    raise AssertionError(f'Subject overlap detected: {subject_overlap}')

subject_split = {sid: 'train' for sid in train_subject_ids}
subject_split.update({sid: 'validation' for sid in validation_subject_ids})

print(f'Train subjects: {len(train_subject_ids)}')
print(f'Validation subjects: {len(validation_subject_ids)}')
print(f'Overlap count: {overlap_count}')

In [ ]:
# 7) Materialize frozen artifacts
results_dir.mkdir(parents=True, exist_ok=True)

baseline_cohort_full_path = results_dir / 'baseline_cohort_full.csv'
baseline_model_table_path = results_dir / 'baseline_model_table.csv'
split_subjects_path = results_dir / 'split_subjects.json'
cohort_summary_path = results_dir / 'cohort_summary.json'

full_columns = list(raw_columns) + ['cognitive_impairment']
with baseline_cohort_full_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=full_columns)
    writer.writeheader()
    for row in baseline_rows:
        writer.writerow({col: row.get(col, '') for col in full_columns})

model_columns = candidate_predictors + ['cognitive_impairment']
with baseline_model_table_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=model_columns)
    writer.writeheader()
    for row in baseline_model_rows:
        writer.writerow({col: row.get(col, '') for col in model_columns})

split_payload = {
    'train_subject_ids': train_subject_ids,
    'validation_subject_ids': validation_subject_ids,
    'train_subject_count': len(train_subject_ids),
    'validation_subject_count': len(validation_subject_ids),
    'overlap_count': overlap_count,
}
with split_subjects_path.open('w', encoding='utf-8') as f:
    json.dump(split_payload, f, indent=2, sort_keys=True)
    f.write('\n')

baseline_positive_count = sum(int(row['cognitive_impairment']) for row in baseline_rows)
baseline_negative_count = len(baseline_rows) - baseline_positive_count
baseline_positive_rate = baseline_positive_count / len(baseline_rows)

missing_counts_retained_predictors = {}
for predictor in candidate_predictors:
    missing_counts_retained_predictors[predictor] = sum(
        1 for row in baseline_rows if str(row.get(predictor, '')).strip() == ''
    )

split_rows = {'train': [], 'validation': []}
for row in baseline_rows:
    sid = row['Subject.ID']
    split_rows[subject_split[sid]].append(row)

def _class_stats(rows_subset):
    pos = sum(int(r['cognitive_impairment']) for r in rows_subset)
    neg = len(rows_subset) - pos
    rate = (pos / len(rows_subset)) if rows_subset else 0.0
    return pos, neg, rate

train_pos, train_neg, train_rate = _class_stats(split_rows['train'])
val_pos, val_neg, val_rate = _class_stats(split_rows['validation'])

cohort_summary_payload = {
    'raw_row_count': len(rows),
    'baseline_row_count': len(baseline_rows),
    'raw_subject_count': len(raw_subject_ids),
    'label_positive_count': baseline_positive_count,
    'label_negative_count': baseline_negative_count,
    'label_positive_rate': baseline_positive_rate,
    'missing_counts_retained_predictors': missing_counts_retained_predictors,
    'train_class_balance': {
        'row_count': len(split_rows['train']),
        'positive_count': train_pos,
        'negative_count': train_neg,
        'positive_rate': train_rate,
    },
    'validation_class_balance': {
        'row_count': len(split_rows['validation']),
        'positive_count': val_pos,
        'negative_count': val_neg,
        'positive_rate': val_rate,
    },
}

with cohort_summary_path.open('w', encoding='utf-8') as f:
    json.dump(cohort_summary_payload, f, indent=2, sort_keys=True)
    f.write('\n')

# Gates required before Step 3
if overlap_count != 0:
    raise AssertionError('Leakage gate failed: overlap_count must be 0.')

if any(col in model_columns for col in locked_exclusions):
    raise AssertionError('Leakage gate failed: excluded columns present in baseline_model_table.csv')

label_domain_baseline = {int(row['cognitive_impairment']) for row in baseline_rows}
if label_domain_baseline != {0, 1}:
    raise AssertionError(f'Label check failed for baseline cohort: {label_domain_baseline}')

if any(str(row.get('cognitive_impairment', '')).strip() == '' for row in baseline_rows):
    raise AssertionError('Label check failed: missing cognitive_impairment values present.')

if len(baseline_rows) != len({row['Subject.ID'] for row in baseline_rows}):
    raise AssertionError('Cohort integrity failed: duplicate Subject.ID in baseline cohort.')

run_date = date.today().isoformat()
docs_text = f"""# Cohort Construction (Step 2A)

Date: {run_date}  
Notebook: `notebooks/02_cohort_construction.ipynb`  
Source data: `data/oasis_dementia_mri_df.csv`

## Cohort Rule and Tie-Break Policy

Baseline cohort includes exactly one row per `Subject.ID`, selected with key priority:
1. minimum `Visit`
2. tie-break minimum `MR.Delay`
3. tie-break lexical `MRI.ID`

Resulting baseline size: **{len(baseline_rows)} subjects** (from {len(rows)} raw rows, {len(raw_subject_ids)} unique subjects).

## Locked Feature Policy

Excluded columns for main baseline model:
- `CDR`, `Group`, `MMSE`, `Subject.ID`, `MRI.ID`, `rownames`, `Hand`, `Visit`, `MR.Delay`

Retained baseline v1 predictors:
- `Age`, `Gender`, `EDUC`, `SES`, `eTIV`, `nWBV`, `ASF`

Model label:
- `cognitive_impairment = int(CDR > 0)`

## Split Method and Leakage Assertion

Deterministic split at **subject level only**:
- bucket = `sha256(subject_id) % 10`
- train if bucket `< 8`, validation otherwise

Leakage assertions:
- subject overlap count: **{overlap_count}**
- excluded columns absent from `baseline_model_table.csv`: **PASS**
- `cognitive_impairment` domain and missingness checks: **PASS**

## Train/Validation Class Balance

| Split | Rows | Positive | Negative | Positive Rate |
|---|---:|---:|---:|---:|
| Train | {len(split_rows['train'])} | {train_pos} | {train_neg} | {train_rate:.4f} |
| Validation | {len(split_rows['validation'])} | {val_pos} | {val_neg} | {val_rate:.4f} |

## Frozen Artifacts

- `results/cohort/baseline_cohort_full.csv`
- `results/cohort/baseline_model_table.csv`
- `results/cohort/split_subjects.json`
- `results/cohort/cohort_summary.json`
"""
docs_path.write_text(docs_text, encoding='utf-8')

print('Wrote:', baseline_cohort_full_path)
print('Wrote:', baseline_model_table_path)
print('Wrote:', split_subjects_path)
print('Wrote:', cohort_summary_path)
print('Wrote:', docs_path)